# Galaxy10 final split review (Member 3)
This notebook is **review-only**. It does not approve or change your split, start training, or overwrite the HDF5. It displays the flagged constant-value image and verifies candidate corrected partition membership. Run in the Kaggle session containing your 2.73 GB Galaxy10 file and M3.2 handoff.


In [ ]:
from pathlib import Path
import hashlib, json, numpy as np, h5py, matplotlib.pyplot as plt
DATA = Path('/kaggle/working/galaxy_data/Galaxy10_DECals_NoDuplicated.h5')
CANDIDATE = Path('/kaggle/working/member3_corrected_split')
assert DATA.is_file(), f'Missing Galaxy10: {DATA}'
assert (CANDIDATE / 'corrected_split_indices.npz').is_file(), 'Attach/extract the M3.2 corrected split first'
print('Dataset:',DATA, 'size GB:',round(DATA.stat().st_size/1e9,3))


## Inspect image 11475
The audit reports a constant-value image at index 11475. Verify with the real dataset and save the figure for team review.


In [ ]:
with h5py.File(DATA,'r') as f:
    im=f['images'][11475]
    label=int(f['ans'][11475])
    ra=float(f['ra'][11475]); dec=float(f['dec'][11475])
print({'index':11475,'label':label,'shape':im.shape,'min':int(im.min()),'max':int(im.max()),'unique_values':len(np.unique(im)),'ra':ra,'dec':dec})
fig, ax=plt.subplots(figsize=(5,5)); ax.imshow(im); ax.axis('off'); ax.set_title('Galaxy10 index 11475: inspect for corruption')
fig.savefig('/kaggle/working/member3_image_11475.png',dpi=160,bbox_inches='tight'); plt.show()


## Check corrected NPZ partition assignments and class counts
This verifies reproducibility and class membership, but not all possible galaxy leakage.


In [ ]:
with np.load(CANDIDATE/'corrected_split_indices.npz', allow_pickle=False) as z:
    ix={k:z[k] for k in z.files}
partition={int(i):k for k in ['pool','val','test'] for i in ix[k]}
assert len(partition)==17675
assert len(set(ix['pool']) & set(ix['val']))==0
assert len(set(ix['pool']) & set(ix['test']))==0
assert len(set(ix['val']) & set(ix['test']))==0
assert set(ix['lab']).issubset(ix['pool'])
assert set(ix['unl_pool']).issubset(ix['pool'])
assert set(ix['lab']).isdisjoint(ix['unl_pool'])
print('Image 11475 partition:',partition[11475]); print('In unlabelled:',11475 in set(ix['unl_pool']))
for a,b in [(17475,17332),(8031,9076),(16062,17231)]:
    print(a,partition[a],b,partition[b]); assert partition[a]==partition[b]
with h5py.File(DATA,'r') as f: y=f['ans'][:]
for k in ['pool','lab','unl_pool','val','test']:
    print(k,len(ix[k]),np.bincount(y[ix[k]].astype(int),minlength=10).tolist())


## Check SHA-256 against documented source file
This reads the entire HDF5 (~2.7 GB); the result must equal the manifest and the value below.


In [ ]:
EXPECTED='a75906f8e3eae0150e3c9de48c7dd74faae3a823130e0030d27d6866a8ffbed4'
h=hashlib.sha256()
with DATA.open('rb') as fp:
    for chunk in iter(lambda:fp.read(8*1024*1024),b''):h.update(chunk)
print('Actual SHA-256:',h.hexdigest()); assert h.hexdigest()==EXPECTED
print('PASS: expected dataset version')


## Record a review decision (do NOT silently approve)
Run with a text decision after actually inspecting the pixels. If image 11475 should be excluded, request that Member 2 implement a versioned exclusion policy and rerun sampler/manifests; the original HDF5 remains unchanged.


In [ ]:
import json, datetime
# Replace these placeholders only AFTER the group reviews the findings.
blank_image_decision = 'PENDING'  # 'RETAIN' or 'EXCLUDE'
blank_image_reason = 'Not yet approved by team'
conservative_overlap_policy_approved = False
record={'date_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'dataset_sha256':EXPECTED,
        'image_11475_decision':blank_image_decision,
        'image_11475_reason':blank_image_reason,
        'geometric_grouping_256px_approved':conservative_overlap_policy_approved,
        'review_status':'APPROVED' if blank_image_decision in ('RETAIN','EXCLUDE') and conservative_overlap_policy_approved else 'PENDING'}
p=Path('/kaggle/working/member3_split_review_decision.json');p.write_text(json.dumps(record,indent=2));print(p,record['review_status'])
